# EU Market Pulse — Data Cleaning & Preparation

## Project Overview

This notebook prepares the raw datasets used to answer the central business 
question of this project: **"Where should a remote-first company expand or 
hire in Europe, based on cost of living and inflation trends?"**

## Data Sources

All data is sourced from Eurostat's public API (no authentication required):

| Dataset | Eurostat Code | Description | Period |
|---|---|---|---|
| HICP Inflation Rate | `tec00118` | Annual average rate of change in consumer prices | 2014–2025 |
| Price Level Indices | `tec00120` | Household consumption price level vs. EU-27 average (100 = EU average) | 2014–2025 |
| House Price Index | `tipsho10` | Deflated house price index, 2015=100 | 2016–2025 |

## Country Scope

The analysis focuses on 19 countries relevant to the target market (remote-first 
companies in the US, UK, Ireland, and Switzerland considering European expansion):

- **Anchor markets:** United Kingdom, Switzerland, Ireland
- **Western Europe:** Germany, France, Netherlands, Austria, Belgium
- **Southern Europe:** Portugal, Spain, Italy, Greece
- **Eastern Europe:** Poland, Romania, Hungary, Czechia
- **Nordics:** Sweden, Denmark, Finland

**Known data limitation:** the House Price Index (`tipsho10`) does not cover 
the United Kingdom or Switzerland. These two countries will show `NULL` values 
in the housing dimension but remain fully covered in the inflation and price 
level datasets.

## Goal of This Notebook

1. Load the three raw datasets
2. Inspect structure and validate integrity
3. Clean and standardize (drop irrelevant columns, rename, filter countries)
4. Export clean, analysis-ready tables to `data/processed/`

## 1. Initial Inspection

Before any cleaning, we confirm that all three files loaded correctly and 
check their raw structure — shape, column names, and a sample of rows.

In [1]:
import pandas as pd

# Load raw datasets exactly as downloaded from Eurostat, no modifications
df_inflation = pd.read_csv('../data/raw/tec00118_hicp_inflation.csv')
df_price_levels = pd.read_csv('../data/raw/tec00120_price_levels.csv')
df_house_prices = pd.read_csv('../data/raw/tipsho10_house_prices.csv')

datasets = {
    'Inflation Rate (tec00118)': df_inflation,
    'Price Levels (tec00120)': df_price_levels,
    'House Prices (tipsho10)': df_house_prices
}

for name, df in datasets.items():
    print(f"--- {name} ---")
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print()

--- Inflation Rate (tec00118) ---
Shape: (487, 10)
Columns: ['DATAFLOW', 'LAST UPDATE', 'freq', 'unit', 'coicop18', 'geo', 'TIME_PERIOD', 'OBS_VALUE', 'OBS_FLAG', 'CONF_STATUS']

--- Price Levels (tec00120) ---
Shape: (516, 10)
Columns: ['DATAFLOW', 'LAST UPDATE', 'freq', 'indic_ppp', 'ppp_cat18', 'geo', 'TIME_PERIOD', 'OBS_VALUE', 'OBS_FLAG', 'CONF_STATUS']

--- House Prices (tipsho10) ---
Shape: (300, 9)
Columns: ['DATAFLOW', 'LAST UPDATE', 'freq', 'unit', 'geo', 'TIME_PERIOD', 'OBS_VALUE', 'OBS_FLAG', 'CONF_STATUS']



## 2. Data Cleaning

For each dataset, we will:

1. Keep only the relevant columns: `geo` (country), `TIME_PERIOD` (year), 
   `OBS_VALUE` (metric value), and `OBS_FLAG` (data quality flag)
2. Filter to the 19 countries defined in scope
3. Rename columns to clear, consistent names
4. Sort by country and year for readability

Metadata columns (`DATAFLOW`, `LAST UPDATE`, `freq`, `unit`, `coicop18`, 
`indic_ppp`, `ppp_cat18`, `CONF_STATUS`) are dropped — they are constant across 
all rows in each file (already filtered at the source) and carry no analytical value.

In [2]:
# Country scope: 19 countries relevant to the target market
COUNTRIES = [
    'United Kingdom', 'Switzerland', 'Ireland',
    'Germany', 'France', 'Netherlands', 'Austria', 'Belgium',
    'Portugal', 'Spain', 'Italy', 'Greece',
    'Poland', 'Romania', 'Hungary', 'Czechia',
    'Sweden', 'Denmark', 'Finland'
]

print(f"Total countries in scope: {len(COUNTRIES)}")

Total countries in scope: 19


In [3]:
# Check exact naming for potentially ambiguous countries
for df, name in [(df_inflation, 'inflation'), (df_price_levels, 'price_levels'), (df_house_prices, 'house_prices')]:
    matches = df[df['geo'].str.contains('Czech|United King', na=False)]['geo'].unique()
    print(f"{name}: {matches}")

inflation: ['Czechia' 'United Kingdom']
price_levels: ['Czechia' 'United Kingdom']
house_prices: ['Czechia']


## 3. Filtering, Renaming and Sorting

With country naming confirmed as consistent across all three sources, we now:

1. Filter each dataset to the 19 countries in scope
2. Select and rename the relevant columns
3. Sort by country and year

A validation check confirms that all 19 countries are present after filtering — 
any missing country would indicate a naming mismatch that needs correction.

In [4]:
def clean_dataset(df, value_name):
    """Filter to scoped countries, select/rename columns, and sort."""
    df_clean = df[df['geo'].isin(COUNTRIES)].copy()
    df_clean = df_clean[['geo', 'TIME_PERIOD', 'OBS_VALUE', 'OBS_FLAG']]
    df_clean = df_clean.rename(columns={
        'geo': 'country',
        'TIME_PERIOD': 'year',
        'OBS_VALUE': value_name,
        'OBS_FLAG': 'flag'
    })
    df_clean = df_clean.sort_values(['country', 'year']).reset_index(drop=True)
    return df_clean

inflation_clean = clean_dataset(df_inflation, 'inflation_rate')
price_levels_clean = clean_dataset(df_price_levels, 'price_level_index')
house_prices_clean = clean_dataset(df_house_prices, 'house_price_index')

# Validation: confirm all countries in scope are present in each cleaned dataset
total_countries = len(COUNTRIES)
for name, df in [('inflation_clean', inflation_clean), 
                  ('price_levels_clean', price_levels_clean), 
                  ('house_prices_clean', house_prices_clean)]:
    found = df['country'].nunique()
    missing = set(COUNTRIES) - set(df['country'].unique())
    print(f"{name}: {found}/{total_countries} countries found. Missing: {missing if missing else 'none'}")

inflation_clean: 19/19 countries found. Missing: none
price_levels_clean: 19/19 countries found. Missing: none
house_prices_clean: 17/19 countries found. Missing: {'Switzerland', 'United Kingdom'}


## 4. Data Quality Checks

Before merging the three cleaned tables, we run a few standard checks:

1. **Duplicates** — confirm no duplicate (country, year) combinations
2. **Data types** — confirm `year` is integer and the value column is numeric
3. **Missing values and flags** — check for `NaN` in the value column, and 
   summarize how many rows carry an Eurostat quality flag (e.g. estimated, 
   provisional, break in series)

In [5]:
for name, df, value_col in [
    ('inflation_clean', inflation_clean, 'inflation_rate'),
    ('price_levels_clean', price_levels_clean, 'price_level_index'),
    ('house_prices_clean', house_prices_clean, 'house_price_index')
]:
    print(f"--- {name} ---")
    print(f"Duplicate (country, year) pairs: {df.duplicated(subset=['country', 'year']).sum()}")
    print(f"Dtypes:\n{df.dtypes}")
    print(f"Missing values in '{value_col}': {df[value_col].isna().sum()}")
    print(f"Flag value counts:\n{df['flag'].value_counts(dropna=False)}")
    print()

--- inflation_clean ---
Duplicate (country, year) pairs: 0
Dtypes:
country            object
year                int64
inflation_rate    float64
flag              float64
dtype: object
Missing values in 'inflation_rate': 0
Flag value counts:
flag
NaN    222
Name: count, dtype: int64

--- price_levels_clean ---
Duplicate (country, year) pairs: 0
Dtypes:
country               object
year                   int64
price_level_index    float64
flag                 float64
dtype: object
Missing values in 'price_level_index': 0
Flag value counts:
flag
NaN    228
Name: count, dtype: int64

--- house_prices_clean ---
Duplicate (country, year) pairs: 0
Dtypes:
country               object
year                   int64
house_price_index    float64
flag                  object
dtype: object
Missing values in 'house_price_index': 0
Flag value counts:
flag
NaN    158
e        9
p        2
ep       1
Name: count, dtype: int64



### Data Quality Summary

All three datasets passed validation with no duplicates, no missing values, 
and correct data types. 

`inflation_clean` and `price_levels_clean` contain no quality flags. 
`house_prices_clean` has 12 flagged observations (9 estimated, 2 provisional, 
1 estimated+provisional) — mostly recent years (2024–2025) where Eurostat has 
not yet finalized figures. These values are retained in the analysis, as 
excluding them would bias the most recent trend.

## 5. Merging the Three Datasets

We merge the three cleaned tables into a single wide-format table, joined on 
`country` and `year`. Since `house_prices_clean` starts in 2016 (vs. 2014 for 
the other two) and excludes Switzerland/UK, we use a **left join** anchored 
on `inflation_clean` — the most complete dataset — so no country-year 
combination is lost. Missing house price data will appear as `NaN`, which 
is expected and documented.

In [6]:
merged = inflation_clean.drop(columns='flag').merge(
    price_levels_clean.drop(columns='flag'),
    on=['country', 'year'],
    how='left'
).merge(
    house_prices_clean.rename(columns={'flag': 'house_price_flag'}),
    on=['country', 'year'],
    how='left'
)

print(merged.shape)
merged.head(10)

(222, 6)


,country,year,inflation_rate,price_level_index,house_price_index,house_price_flag
0,Austria,2014,1.5,109.3,NaN,NaN
1,Austria,2015,0.8,109.3,NaN,NaN
2,Austria,2016,1.0,110.4,105.27,NaN
3,Austria,2017,2.2,111.6,108.56,NaN
4,Austria,2018,2.1,112.1,112.62,NaN
5,Austria,2019,1.5,113.2,117.33,NaN
6,Austria,2020,1.4,111.7,124.93,NaN
7,Austria,2021,2.8,108.5,136.59,NaN
8,Austria,2022,8.6,107.9,141.47,NaN
9,Austria,2023,7.7,111.5,127.10,NaN


## 6. Final Validation

Before exporting, we confirm that missing values in `house_price_index` are 
fully explained by the two known limitations: years before 2016 (series start) 
and the excluded countries (Switzerland, United Kingdom). Any missing value 
outside these two cases would indicate an unexpected data quality issue.

In [7]:
missing_house = merged[merged['house_price_index'].isna()]

expected_missing = (
    (missing_house['year'] < 2016) | 
    (missing_house['country'].isin(['Switzerland', 'United Kingdom']))
)

print(f"Total missing house_price_index rows: {len(missing_house)}")
print(f"Explained by known limitations: {expected_missing.sum()}")
print(f"Unexpected missing rows: {(~expected_missing).sum()}")

if (~expected_missing).sum() > 0:
    print(missing_house[~expected_missing])

Total missing house_price_index rows: 52
Explained by known limitations: 52
Unexpected missing rows: 0


## 7. Export Clean Dataset

The final merged and validated dataset is exported to `data/processed/` as 
the single source of truth for the next stages of the project (SQL analysis, 
cloud storage, forecasting, and dashboard).

In [8]:
merged.to_csv('../data/processed/eu_market_pulse_clean.csv', index=False)

print(f"Exported: {merged.shape[0]} rows, {merged.shape[1]} columns")
print(f"Countries: {merged['country'].nunique()}")
print(f"Year range: {merged['year'].min()}–{merged['year'].max()}")

Exported: 222 rows, 6 columns
Countries: 19
Year range: 2014–2025
